### How to build a GRN with pySCENIC — no Docker or Snakemake required!

Step 0: Imports and setup

In [ ]:
import os, glob, pickle
import pandas as pd
from dask.diagnostics import ProgressBar

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2          # or genie3

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

Phase I: GRN inference (co-expression modules)

In [ ]:
# Load expression matrix: rows=cells, columns=genes
ex_matrix = pd.read_csv("expression.tsv", sep='\t', header=0, index_col=0).T

# Load TF names
tf_names = load_tf_names("tf_list.txt")

# Run GRNBoost2 to get TF-target adjacencies
adjacencies = grnboost2(ex_matrix, tf_names=tf_names, verbose=True)

# Convert adjacencies to co-expression modules (candidate regulons)
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))

grnboost2 returns a DataFrame with columns TF, target, importance. pySCENIC:131-162

modules_from_adjacencies applies several thresholding strategies (percentile cutoffs, top-N targets, top-N regulators) and uses Pearson correlation to split activating vs. repressing modules. pySCENIC:267-299

Phase II: Prune with cisTarget (motif enrichment → regulons)


In [ ]:
# Load ranking databases
db_fnames = glob.glob("path/to/databases/*.genes_vs_motifs.rankings.feather")
dbs = [RankingDatabase(fname=f, name=os.path.splitext(os.path.basename(f))[0])
       for f in db_fnames]

# Prune modules: keep only targets with cis-regulatory motif support
with ProgressBar():
    df = prune2df(dbs, modules, "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl")

# Convert enriched motif table to Regulon objects
regulons = df2regulons(df)

# Optionally save
df.to_csv("motifs.csv")
with open("regulons.pkl", "wb") as f:
    pickle.dump(regulons, f)

Phase III: AUCell — score regulon activity per cell

In [ ]:
auc_mtx = aucell(ex_matrix, regulons, num_workers=4)
# auc_mtx: DataFrame of shape (n_cells x n_regulons)
auc_mtx.to_csv("auc_matrix.csv")

### How to build a GRN with pySCENIC — with pyGS!